# Notebook 02 — Data Pipeline

**Purpose:** Run the full data pipeline for all three indices and verify correctness.

Sections:
1. Bootstrap
2. Run pipeline for each index × feature set
3. Anti-leakage assertions (displayed as cell output)
4. Scaler verification (fit on train only)
5. Data quality report
6. Save summary table


## 0. Bootstrap

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_URL = "https://github.com/stephugm/incremental-learning.git"
FORCE_DOWNLOAD = False

def detect_env():
    if "google.colab" in sys.modules: return "colab"
    if os.path.exists("/kaggle/input"): return "kaggle"
    return "local"

ENV = detect_env()
print(f"Environment: {ENV}")

if ENV in ("colab", "kaggle"):
    REPO = (Path("/content/incremental-learning") if ENV == "colab"
            else Path("/kaggle/working/incremental-learning"))
    if not REPO.exists():
        subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)

    # ── Step 1: install requirements (tvDatafeed sudah dipisah) ──────────
    subprocess.run(["pip", "install", "-q", "-r",
                    str(REPO / "requirements.txt")], check=False)

    # ── Step 2: tvDatafeed best-effort dari GitHub ────────────────────────
    print("Installing tvDatafeed from GitHub ...")
    try:
        r = subprocess.run(
            ["pip", "install", "-q",
             "git+https://github.com/rongardF/tvdatafeed.git"],
            capture_output=True, timeout=120,
        )
        from tvDatafeed import TvDatafeed  # noqa: F401
        print("tvDatafeed : OK  -> target 6000 bar dapat dicapai")
    except Exception as _e:
        print(f"tvDatafeed : TIDAK TERSEDIA ({_e})")
        print("  -> fallback yfinance, hanya ~1500 bar (limit 60 hari)")
        print("  -> n_bars=6000 di config TIDAK akan tercapai via yfinance")

else:
    REPO = Path().resolve()
    while not (REPO / "configs" / "config.yaml").exists() and REPO != REPO.parent:
        REPO = REPO.parent

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print(f"Repo root: {REPO}")
# autoreload: best-effort, skip gracefully on Python 3.12+ if imp removed
try:
    _ip = get_ipython()  # noqa: F821
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
except Exception as _ar_e:
    print(f'autoreload skipped: {_ar_e}')

In [ ]:
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.paths import Paths
from src.data import build_dataset

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

paths = Paths()
with open(paths.configs / "config.yaml") as f:
    cfg = yaml.safe_load(f)

INDICES = cfg["data"]["indices"]
FEATURE_SETS = list(cfg["data"]["feature_sets"].keys())  # ["univariate", "multivariate"]
LAG = cfg["data"]["lag"]
TRAIN_RATIO = cfg["data"]["split"]["train"]
N_BARS = cfg["data"]["n_bars"]
EMA5_PERIOD = cfg["indicators"]["ema5_period"]

print(f"Indices: {[i['symbol'] for i in INDICES]}")
print(f"Feature sets: {FEATURE_SETS}")
print(f"Lag: {LAG} | Train ratio: {TRAIN_RATIO} | N bars target: {N_BARS}")

## 1. Run full pipeline for all indices × feature sets

In [ ]:
datasets = {}  # key: (symbol, feature_set)

for idx_cfg in INDICES:
    sym = idx_cfg["symbol"]
    for fset in FEATURE_SETS:
        print(f"\n{'='*55}")
        print(f"Processing: {sym} | {fset}")
        print("="*55)
        ds = build_dataset(
            symbol=sym,
            exchange=idx_cfg["exchange"],
            tz=idx_cfg["tz"],
            feature_set=fset,
            paths=paths,
            n_bars=N_BARS,
            lag=LAG,
            train_ratio=TRAIN_RATIO,
            force_download=FORCE_DOWNLOAD,
            ema5_period=EMA5_PERIOD,
        )
        datasets[(sym, fset)] = ds

print("\n✓ All datasets built.")

## 2. Anti-leakage assertions

These assertions must ALL pass before any model training begins.

- Train timestamps must end strictly before test timestamps start
- Scaler must have been fit only on train data (min/max should match train range)

In [ ]:
print("Anti-leakage assertions:")
all_pass = True

for (sym, fset), ds in datasets.items():
    ts_train = ds["ts_train"]
    ts_test  = ds["ts_test"]

    # Check 1: chronological ordering
    assert ts_train[-1] < ts_test[0], f"FAIL: {sym}/{fset} — train/test overlap!"

    # Check 2: scaler data range — train y should be in [0, 1] after scaling
    y_train_s = ds["y_train"]
    assert y_train_s.min() >= -0.01 and y_train_s.max() <= 1.01, \
        f"FAIL: {sym}/{fset} — scaled y_train out of [0,1]?"

    # Check 3: test y may exceed [0,1] (valid — prices can be outside train range)
    # but we warn if it's extreme
    y_test_s = ds["y_test"]
    if y_test_s.min() < -0.5 or y_test_s.max() > 1.5:
        print(f"  WARN: {sym}/{fset} — test y outside [-0.5, 1.5] after scaling. "
              "Check for large price regime shift.")

    gap_days = (ts_test[0] - ts_train[-1]).days
    print(f"  ✓ {sym:6s}/{fset:12s} | "
          f"train ends {ts_train[-1].date()} | "
          f"test starts {ts_test[0].date()} | "
          f"gap {gap_days}d")

print(f"\n{'All assertions PASSED ✓' if all_pass else 'Some assertions FAILED ✗'}")

## 3. Dataset summary table

In [ ]:
rows = []
for (sym, fset), ds in datasets.items():
    rows.append({
        "Symbol": sym,
        "Feature set": fset,
        "Features": ds["feature_names"],
        "N train": len(ds["X_train"]),
        "N test": len(ds["X_test"]),
        "Train start": ds["ts_train"][0].date(),
        "Train end": ds["ts_train"][-1].date(),
        "Test start": ds["ts_test"][0].date(),
        "Test end": ds["ts_test"][-1].date(),
        "X shape": str(ds["X_train"].shape),
        "y_train min": f"{ds['y_train_raw'].min():.2f}",
        "y_train max": f"{ds['y_train_raw'].max():.2f}",
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))

# Save
summary_path = paths.results / "pipeline_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"\nSaved → {summary_path}")

## 4. Visualise train/test split for each index

In [ ]:
fig, axes = plt.subplots(len(INDICES), 1, figsize=(14, 3 * len(INDICES)), sharex=False)
if len(INDICES) == 1:
    axes = [axes]

for ax, idx_cfg in zip(axes, INDICES):
    sym = idx_cfg["symbol"]
    # Use univariate dataset (same df_clean for both feature sets)
    ds = datasets[(sym, "univariate")]
    df_clean = ds["df_clean"]

    n_train = len(ds["X_train"])
    n_test  = len(ds["X_test"])
    split_idx = n_train + LAG  # account for window offset

    ax.plot(df_clean.index[:split_idx], df_clean["close"].iloc[:split_idx],
            lw=0.5, color="steelblue", label="Train")
    ax.plot(df_clean.index[split_idx:], df_clean["close"].iloc[split_idx:],
            lw=0.5, color="tomato", label="Test")
    ax.axvline(df_clean.index[split_idx], color="black", ls="--", lw=0.8)
    ax.set_title(f"{sym} — train/test split (70/30)")
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(paths.figures / "train_test_split.png", dpi=150)
plt.show()

## 5. Scaler verification

In [ ]:
print("Scaler verification (fit on train only):")
print("-" * 60)
for (sym, fset), ds in datasets.items():
    sy = ds["scaler_y"]
    # scaler_y was fit on y_train_raw
    train_min = ds["y_train_raw"].min()
    train_max = ds["y_train_raw"].max()
    scaler_min = float(sy.data_min_[0])
    scaler_max = float(sy.data_max_[0])
    match = abs(train_min - scaler_min) < 0.01 and abs(train_max - scaler_max) < 0.01
    status = "✓" if match else "✗ MISMATCH"
    print(f"  {status} {sym:6s}/{fset:12s} | "
          f"train price [{train_min:.2f}, {train_max:.2f}] | "
          f"scaler [{scaler_min:.2f}, {scaler_max:.2f}]")

## 6. EMA5 causal implementation check

Verify that EMA5 at time $t$ depends only on closes at $t, t-1, t-2, \ldots$
(no look-ahead). We do this by comparing the incremental computation
against a forward-only rolling calculation.

In [ ]:
from src.data import IncrementalEMA

# Take SPX univariate dataset
ds = datasets[("SPX", "univariate")]
df_clean = ds["df_clean"]

close = df_clean["close"].values
ema5_incremental = df_clean["ema5"].values

# Manual recursive check on first 20 bars
beta = 2 / (5 + 1)
manual = [close[0]]
for i in range(1, 20):
    manual.append(beta * close[i] + (1 - beta) * manual[-1])

print("Causal EMA5 check (first 20 bars):")
print(f"{'Bar':>4} {'close':>10} {'incremental':>12} {'manual':>10} {'match':>6}")
for i in range(20):
    ok = abs(ema5_incremental[i] - manual[i]) < 1e-4
    print(f"{i:4d} {close[i]:10.4f} {ema5_incremental[i]:12.4f} {manual[i]:10.4f} {'✓' if ok else '✗'}")

all_match = all(abs(ema5_incremental[i] - manual[i]) < 1e-4 for i in range(20))
print(f"\nAll match: {'✓ YES' if all_match else '✗ NO'}")

## 7. Processed files on disk

In [ ]:
import os

print(f"Processed directory: {paths.processed}")
for f in sorted(paths.processed.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40} {size_kb:8.1f} KB")

print(f"\nRaw directory: {paths.raw}")
for f in sorted(paths.raw.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<40} {size_kb:8.1f} KB")